# Tech Challenge Fase 2
## 05.3 — Data Quality Gold

Valida os produtos Gold antes da liberação para Power BI e Machine Learning.

Principais verificações:

- unicidade por município/UF e ano;
- cobertura de metas;
- indicadores entre 0 e 100;
- completude da base de IA;
- validade do target;
- persistência de rejeitados;
- bloqueio do pipeline quando houver regra crítica reprovada.

## 1. Imports

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, LongType, DoubleType,
    BooleanType
)

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

GOLD_PATH = config["paths"]["gold_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

QUALITY_ROOT_PATH = f"{LOG_PATH}/data_quality"
QUALITY_GOLD_PATH = f"{QUALITY_ROOT_PATH}/gold"
QUALITY_DETAILS_PATH = f"{QUALITY_GOLD_PATH}/details"
QUALITY_SUMMARY_PATH = f"{QUALITY_GOLD_PATH}/summary"
QUALITY_REJECTED_PATH = f"{QUALITY_ROOT_PATH}/rejected/gold"
QUALITY_HISTORY_PATH = f"{QUALITY_ROOT_PATH}/history/gold"

for path in [
    QUALITY_GOLD_PATH,
    QUALITY_DETAILS_PATH,
    QUALITY_SUMMARY_PATH,
    QUALITY_REJECTED_PATH,
    QUALITY_HISTORY_PATH
]:
    dbutils.fs.mkdirs(path)

print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Leitura das regras Gold

In [0]:
df_quality_metadata = spark.read.parquet(
    f"{CONFIG_PATH}/quality_metadata"
)

df_gold_rules = (
    df_quality_metadata
    .filter(
        (F.col("layer") == "gold")
        & F.col("enabled")
    )
    .orderBy("dataset", "rule_id")
)

gold_rules = [
    row.asDict()
    for row in df_gold_rules.collect()
]

display(df_gold_rules)
print("Regras Gold ativas:", len(gold_rules))

## 4. Mapeamento dos produtos Gold

In [0]:
gold_dataset_paths = {
    "gold_alunos": {
        "years": [2023, 2024, 2025],
        "template": f"{GOLD_PATH}/alunos/ano={{ano}}/GOLD_ALUNOS_{{ano}}.csv"
    },
    "gold_municipios": {
        "years": [2023, 2024, 2025],
        "template": f"{GOLD_PATH}/municipios/ano={{ano}}/GOLD_MUNICIPIOS_{{ano}}.csv"
    },
    "gold_estados": {
        "years": [2023, 2024, 2025],
        "template": f"{GOLD_PATH}/estados/ano={{ano}}/GOLD_ESTADOS_{{ano}}.csv"
    },
    "gold_machine_learning": {
        "years": [0],
        "template": f"{GOLD_PATH}/base_modelo_ia/BASE_MODELO_IA_COMPLETA.csv"
    }
}

column_aliases = {
    "ANO": ["ANO", "ano"],
    "CO_MUNICIPIO": ["CO_MUNICIPIO", "id_municipio"],
    "CO_UF": ["CO_UF", "CD_UF", "co_uf"],
    "PC_ALUNO_ALFABETIZADO": ["PC_ALUNO_ALFABETIZADO", "taxa_alfabetizacao"],
    "META_FINAL_2030": ["META_FINAL_2030", "meta_2030"],
    "registro_completo_modelo": ["registro_completo_modelo"],
    "risco_nao_atingir_meta": ["risco_nao_atingir_meta"]
}

## 5. Funções auxiliares

In [0]:
def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def split_columns(value):
    return [
        item.strip()
        for item in str(value or "").split(",")
        if item.strip()
    ]


def resolve_column(expected, available):
    if expected in available:
        return expected

    for alias in column_aliases.get(expected, []):
        if alias in available:
            return alias

    return None


def calculate_status(invalid_percent, tolerance_percent):
    if invalid_percent <= tolerance_percent:
        return "APROVADO"

    attention_limit = max(
        tolerance_percent * 2,
        tolerance_percent + 1.0
    )

    if invalid_percent <= attention_limit:
        return "ATENCAO"

    return "REPROVADO"


def read_gold_csv(path):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

    for column in df.columns:
        normalized = column.strip()

        if normalized != column:
            df = df.withColumnRenamed(
                column,
                normalized
            )

    return df


def null_like(column):
    return (
        F.col(column).isNull()
        | (F.trim(F.col(column).cast("string")) == "")
        | F.lower(
            F.trim(F.col(column).cast("string"))
        ).isin("nan", "none", "<na>", "null")
    )


def numeric_value(column):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.trim(F.col(column).cast("string")),
                "%",
                ""
            ),
            ",",
            "."
        )
        .cast("double")
    )

## 6. Schema dos resultados

In [0]:
schema_result = StructType([
    StructField("layer", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("blocking", BooleanType(), False),
    StructField("tolerance_percent", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("records_evaluated", LongType(), False),
    StructField("invalid_records", LongType(), False),
    StructField("invalid_percent", DoubleType(), False),
    StructField("message", StringType(), True),
    StructField("evaluated_path", StringType(), True),
    StructField("execution_date", StringType(), False),
    StructField("evaluated_at", StringType(), False)
])

## 7. Execução das regras

In [0]:
quality_results = []
rejected_paths = []

for rule in gold_rules:
    dataset = rule["dataset"]
    cfg = gold_dataset_paths.get(dataset)

    if cfg is None:
        continue

    for ano in cfg["years"]:
        path = (
            cfg["template"]
            if ano == 0
            else cfg["template"].format(ano=ano)
        )

        df = None
        available = []
        records = 0
        read_error = ""

        if file_exists(path):
            try:
                df = read_gold_csv(path)
                available = df.columns
                records = df.count()
            except Exception as e:
                read_error = str(e)
        else:
            read_error = "Arquivo Gold não encontrado."

        expected = split_columns(rule["column_name"])
        resolved = []
        missing = []

        for column in expected:
            found = resolve_column(column, available)

            if found:
                resolved.append(found)
            else:
                missing.append(column)

        invalid_df = None
        invalid = 0
        invalid_percent = 0.0
        message = ""
        rule_type = rule["rule_type"]

        if df is None:
            invalid = 1
            invalid_percent = 100.0
            message = f"Arquivo indisponível: {read_error}"

        elif records == 0:
            invalid = 1
            invalid_percent = 100.0
            message = "Arquivo Gold vazio."

        elif missing:
            invalid = records
            invalid_percent = 100.0
            message = "Colunas ausentes: " + ", ".join(missing)

        elif rule_type == "unique":
            duplicates = (
                df.groupBy(*resolved)
                .count()
                .filter(F.col("count") > 1)
            )

            invalid = (
                duplicates
                .agg(
                    F.sum(
                        F.col("count") - 1
                    ).alias("value")
                )
                .first()["value"]
                or 0
            )

            invalid_percent = invalid / records * 100

            if invalid > 0:
                invalid_df = df.join(
                    duplicates.drop("count"),
                    on=resolved,
                    how="inner"
                )

            message = (f"{invalid} registros excedentes em chaves duplicadas para {resolved}. " f"A origem deve ser consolidada antes da persistência Gold.")

        elif rule_type in ["coverage", "not_null"]:
            target = resolved[0]

            invalid_df = df.filter(
                null_like(target)
            )

            invalid = invalid_df.count()
            invalid_percent = invalid / records * 100
            message = f"{invalid} registros sem valor em {target}."

        elif rule_type == "between":
            target = resolved[0]
            value = numeric_value(target)

            invalid_df = df.filter(
                F.col(target).isNotNull()
                & (
                    value.isNull()
                    | ~value.between(0.0, 100.0)
                )
            )

            invalid = invalid_df.count()
            invalid_percent = invalid / records * 100
            message = f"{invalid} registros fora da faixa de 0 a 100."

        elif rule_type == "completeness":
            target = resolved[0]

            invalid_df = df.filter(
                F.col(target).isNull()
                | F.lower(
                    F.trim(
                        F.col(target).cast("string")
                    )
                ).isin("false", "0", "0.0")
            )

            invalid = invalid_df.count()
            invalid_percent = invalid / records * 100
            message = f"{invalid} registros incompletos para modelagem."

        elif rule_type == "allowed_values":
            target = resolved[0]
            allowed = ["0", "1", "0.0", "1.0"]

            invalid_df = df.filter(
                F.col(target).isNotNull()
                & ~F.trim(
                    F.col(target).cast("string")
                ).isin(allowed)
            )

            invalid = invalid_df.count()
            invalid_percent = invalid / records * 100
            message = f"{invalid} registros fora dos valores permitidos."

        else:
            invalid = records
            invalid_percent = 100.0
            message = f"Tipo de regra não suportado: {rule_type}."

        status = calculate_status(
            invalid_percent,
            float(rule["tolerance_percent"])
        )

        if invalid_df is not None and invalid > 0:
            rejected_path = (
                f"{QUALITY_REJECTED_PATH}/"
                f"dataset={dataset}/"
                f"ano={ano}/"
                f"rule_id={rule['rule_id']}/"
                f"execution_date={EXECUTION_DATE}"
            )

            (
                invalid_df
                .withColumn(
                    "_quality_rule_id",
                    F.lit(rule["rule_id"])
                )
                .withColumn(
                    "_quality_reason",
                    F.lit(message)
                )
                .withColumn(
                    "_quality_execution_date",
                    F.lit(EXECUTION_DATE)
                )
                .coalesce(1)
                .write
                .mode("overwrite")
                .format("parquet")
                .option("compression", "snappy")
                .save(rejected_path)
            )

            rejected_paths.append(rejected_path)

        quality_results.append({
            "layer": "gold",
            "dataset": dataset,
            "ano": int(ano),
            "rule_id": rule["rule_id"],
            "rule_name": rule["rule_name"],
            "rule_type": rule_type,
            "column_name": rule["column_name"],
            "severity": rule["severity"],
            "blocking": bool(rule["blocking"]),
            "tolerance_percent": float(rule["tolerance_percent"]),
            "status": status,
            "records_evaluated": int(records),
            "invalid_records": int(invalid),
            "invalid_percent": float(round(invalid_percent, 4)),
            "message": message,
            "evaluated_path": path,
            "execution_date": str(EXECUTION_DATE),
            "evaluated_at": datetime.now().isoformat()
        })

print("Validações executadas:", len(quality_results))
print("Conjuntos de rejeitados:", len(rejected_paths))

## 8. Resultados detalhados

In [0]:
df_quality_results = spark.createDataFrame(
    quality_results,
    schema=schema_result
)

display(
    df_quality_results
    .orderBy(
        "dataset",
        "ano",
        "rule_id"
    )
)

details_path = (
    f"{QUALITY_DETAILS_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_results
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(details_path)
)

print("Detalhes salvos em:", details_path)

## 9. Resumo executivo

In [0]:
df_quality_summary = (
    df_quality_results
    .groupBy(
        "layer",
        "dataset",
        "ano"
    )
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(
            F.when(
                F.col("status") == "APROVADO",
                1
            ).otherwise(0)
        ).alias("approved_rules"),
        F.sum(
            F.when(
                F.col("status") == "ATENCAO",
                1
            ).otherwise(0)
        ).alias("warning_rules"),
        F.sum(
            F.when(
                F.col("status") == "REPROVADO",
                1
            ).otherwise(0)
        ).alias("failed_rules"),
        F.sum(
            F.when(
                (F.col("status") == "REPROVADO")
                & F.col("blocking"),
                1
            ).otherwise(0)
        ).alias("blocking_failed_rules"),
        F.sum("invalid_records").alias(
            "total_invalid_records"
        )
    )
    .withColumn(
        "quality_score_percent",
        F.round(
            F.col("approved_rules")
            / F.col("total_rules")
            * 100,
            2
        )
    )
    .withColumn(
        "dataset_status",
        F.when(
            F.col("blocking_failed_rules") > 0,
            "REPROVADO"
        )
        .when(
            F.col("failed_rules") > 0,
            "ATENCAO"
        )
        .otherwise("APROVADO")
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_quality_summary
    .orderBy("dataset", "ano")
)

summary_path = (
    f"{QUALITY_SUMMARY_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(summary_path)
)

print("Resumo salvo em:", summary_path)

## 10. Histórico

In [0]:
(
    df_quality_results
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("execution_date")
    .save(QUALITY_HISTORY_PATH)
)

print("Histórico atualizado em:", QUALITY_HISTORY_PATH)

## Diagnóstico por origem da reprovação

A tabela abaixo diferencia:

- arquivo ausente;
- coluna ausente;
- duplicidade de granularidade;
- falha de cobertura;
- target inválido.

As regras críticas permanecem bloqueantes.

In [0]:
df_diagnostico_gold = (
    df_quality_results
    .filter(
        F.col("status") != "APROVADO"
    )
    .select(
        "dataset",
        "ano",
        "rule_id",
        "rule_type",
        "severity",
        "blocking",
        "invalid_records",
        "invalid_percent",
        "message",
        "evaluated_path"
    )
    .orderBy(
        "blocking",
        "dataset",
        "ano",
        "rule_id"
    )
)

if df_diagnostico_gold.count() == 0:
    print("Nenhuma reprovação ou alerta na Gold.")
else:
    display(df_diagnostico_gold)

## 11. Diagnóstico e checklist final

In [0]:
df_bloqueios = (
    df_quality_results
    .filter(
        (F.col("status") == "REPROVADO")
        & F.col("blocking")
    )
    .orderBy(
        "dataset",
        "ano",
        "rule_id"
    )
)

blocking_count = df_bloqueios.count()

if blocking_count > 0:
    display(df_bloqueios)

    raise Exception(
        f"Foram encontradas "
        f"{blocking_count} regras "
        f"bloqueantes reprovadas na "
        f"camada Gold. Corrija os "
        f"produtos de dados antes de "
        f"liberar Power BI e IA."
    )

print("Data Quality Gold concluída com sucesso.")
print("Todas as regras críticas foram aprovadas.")

## Resultado esperado

```text
logs/data_quality/gold/details/execution_date=YYYY-MM-DD
logs/data_quality/gold/summary/execution_date=YYYY-MM-DD
logs/data_quality/rejected/gold/
logs/data_quality/history/gold
```

Próximo notebook:

```text
05_4_quality_dashboard
```